In [ ]:
# rende config.py (in notebooks/) importabile anche dai notebook in sottocartelle
import sys, os
_cfg = os.getcwd()
while _cfg != os.path.dirname(_cfg):
    if os.path.exists(os.path.join(_cfg, 'config.py')):
        sys.path.insert(0, _cfg)
        break
    _cfg = os.path.dirname(_cfg)
from config import CONFESS_DATA, BC_DATA, ERA5_ROOT, POST_DATA, WORK_DIR, FIG_DIR, FIG_DIR_2025
import xarray as xr
import glob, os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
plt.rcParams.update({'font.size': 22})
import concurrent.futures
from concurrent.futures import ProcessPoolExecutor
from scipy import stats
import matplotlib.patches as mpatches
import pandas as pd
import logging

In [ ]:
# Configurazione logging
log_file = "03-Figx3_global_bias_tas_seasons.log"
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    handlers=[
        logging.FileHandler(log_file),  # Write to file
    ]
)

In [ ]:
exp_ctrl = 'a1ua'
exp_sens = 'a52o'
var='tas'
era_var='2t'
SAVE_PATH = f'{WORK_DIR}/'

In [ ]:
def main_script(season):
    logging.info(f"Starting main {season}")
    try:
        dset_ctrl = xr.open_mfdataset(f'{SAVE_PATH}/{exp_ctrl}_{var}_lead_*_1x1_global_average_all_members_{season}_1999.nc', concat_dim='lead_time', combine='nested', chunks=-1)
        dset_sens = xr.open_mfdataset(f'{SAVE_PATH}/{exp_sens}_{var}_lead_*_1x1_global_average_all_members_{season}_1999.nc', concat_dim='lead_time', combine='nested', chunks=-1)
        
        dset_ctrl_land = xr.open_mfdataset(f'{SAVE_PATH}/{exp_ctrl}_{var}_lead_*_1x1_global_average_all_members_land_{season}_1999.nc', concat_dim='lead_time', combine='nested', chunks=-1)
        dset_sens_land = xr.open_mfdataset(f'{SAVE_PATH}/{exp_sens}_{var}_lead_*_1x1_global_average_all_members_land_{season}_1999.nc', concat_dim='lead_time', combine='nested', chunks=-1)
        
        dset_ctrl_ocean = xr.open_mfdataset(f'{SAVE_PATH}/{exp_ctrl}_{var}_lead_*_1x1_global_average_all_members_ocean_{season}_1999.nc', concat_dim='lead_time', combine='nested', chunks=-1)
        dset_sens_ocean = xr.open_mfdataset(f'{SAVE_PATH}/{exp_sens}_{var}_lead_*_1x1_global_average_all_members_ocean_{season}_1999.nc', concat_dim='lead_time', combine='nested', chunks=-1)
    
        leads = [0, 1, 2, 3, 4]
        ld = xr.DataArray(leads, dims='lead_time', name='lead_time')
        dset_ctrl['lead_time']=ld
        dset_sens['lead_time']=ld
        dset_ctrl_land['lead_time']=ld
        dset_sens_land['lead_time']=ld
        dset_ctrl_ocean['lead_time']=ld
        dset_sens_ocean['lead_time']=ld
        
        
        em_ctrl = dset_ctrl.mean('member')
        em_sens = dset_sens.mean('member')
        em_ctrl_land = dset_ctrl_land.mean('member')
        em_sens_land = dset_sens_land.mean('member')
        em_ctrl_ocean = dset_ctrl_ocean.mean('member')
        em_sens_ocean = dset_sens_ocean.mean('member')
        
        # Convert the dataset to a DataFrame
        dset_ctrl = dset_ctrl.to_dataframe().reset_index()
        dset_sens = dset_sens.to_dataframe().reset_index()
        dset_ctrl_land = dset_ctrl_land.to_dataframe().reset_index()
        dset_sens_land = dset_sens_land.to_dataframe().reset_index()
        dset_ctrl_ocean = dset_ctrl_ocean.to_dataframe().reset_index()
        dset_sens_ocean = dset_sens_ocean.to_dataframe().reset_index()
    
        plot_figures(season, dset_ctrl, dset_ctrl_land, dset_ctrl_ocean, dset_sens, dset_sens_land, dset_sens_ocean, em_ctrl, em_ctrl_land, em_ctrl_ocean, em_sens, em_sens_land, em_sens_ocean)

        logging.info(f"Bias files written successfully for season {season}")

    except Exception as e:
        logging.exception(f"Error occurred for season {season}: {e}")

In [ ]:
# --- Schema dei boxplot SENS/CTRL, identico a 03-Fig3_global_bias_tas.ipynb
# (annuale): box affiancati (SENS lightcoral a sinistra, CTRL plum a destra) e
# significativita' codificata nella trasparenza del colore: box opaco se SENS e
# CTRL differiscono significativamente a quella posizione (Mann-Whitney U),
# sbiadito altrimenti. Usato sia dai pannelli per lead singolo (plot_figures,
# cella successiva) sia dalle combinazioni di lead year (in fondo).
SIGN_ALPHA = 0.05   # soglia; usa 0.10 per il 90%
ALPHA_SIG = 0.9
ALPHA_NONSIG = 0.25

def significant_labels(df, label_col, labels, value_col=var, alpha=SIGN_ALPHA):
    """Ritorna l'insieme delle etichette (lead o combinazione) in cui SENS e
    CTRL differiscono significativamente (Mann-Whitney U). 'df' e' il
    dataframe combinato con una colonna 'exp' ('SENS'/'CTRL')."""
    sig = set()
    for lab in labels:
        s = df.loc[(df[label_col] == lab) & (df['exp'] == 'SENS'), value_col].dropna()
        c = df.loc[(df[label_col] == lab) & (df['exp'] == 'CTRL'), value_col].dropna()
        if len(s) < 3 or len(c) < 3:
            continue
        if stats.mannwhitneyu(s, c, alternative='two-sided').pvalue < alpha:
            sig.add(lab)
    return sig

def _draw_significance_boxplot(ax, positions, labels, df, label_col, sig_labels,
                                value_col=var, color_sens='lightcoral', color_ctrl='plum',
                                width=0.35, off=0.2):
    """Disegna i box SENS/CTRL affiancati per ciascuna posizione con
    ax.boxplot (non sns.boxplot): patch_artist=True restituisce i patch
    nell'ordine ESATTO della lista di dati passata, a differenza dell'ordine
    dei patch di seaborn con hue (che dipende dalla versione) - necessario
    per essere sicuri di applicare l'alpha al box giusto."""
    sens_data = [df.loc[(df[label_col] == l) & (df['exp'] == 'SENS'), value_col].dropna().values for l in labels]
    ctrl_data = [df.loc[(df[label_col] == l) & (df['exp'] == 'CTRL'), value_col].dropna().values for l in labels]
    alphas = [ALPHA_SIG if l in sig_labels else ALPHA_NONSIG for l in labels]

    for data_list, pos_offset, color in [(sens_data, -off, color_sens), (ctrl_data, off, color_ctrl)]:
        bp = ax.boxplot(data_list, positions=positions + pos_offset, widths=width,
                        patch_artist=True, manage_ticks=False, medianprops=dict(color='black'))
        for patch, a in zip(bp['boxes'], alphas):
            patch.set_facecolor(color)
            patch.set_alpha(a)

    ax.set_xticks(positions)
    ax.set_xticklabels(labels)
    ax.legend(handles=[mpatches.Patch(color=color_sens, label='SENS'),
                       mpatches.Patch(color=color_ctrl, label='CTRL')])

In [ ]:
def _plot_panel(df_sens, df_ctrl, em_sens, em_ctrl, title, outfile):
    """Un pannello (global/land/ocean) per lead singolo, nello stesso schema
    di colori e trasparenze di 03-Fig3_global_bias_tas.ipynb (annuale)."""
    df = pd.concat([df_sens.assign(exp='SENS'), df_ctrl.assign(exp='CTRL')])
    leads = sorted(df['lead_time'].unique())
    positions = np.arange(len(leads))

    fig, ax = plt.subplots(figsize=(12, 6))
    sig = significant_labels(df, 'lead_time', leads)
    _draw_significance_boxplot(ax, positions, leads, df, 'lead_time', sig)

    # medie d'ensemble centrate sul box corrispondente
    ax.plot(positions - 0.2, em_sens[var].values, color='k')                  # SENS (box a sinistra)
    ax.plot(positions + 0.2, em_ctrl[var].values, color='k', linestyle='--')  # CTRL (box a destra)
    ax.set_title(title)

    fig.savefig(outfile, dpi=600, bbox_inches='tight')
    plt.close(fig)

def plot_figures(season, dset_ctrl, dset_ctrl_land, dset_ctrl_ocean, dset_sens, dset_sens_land, dset_sens_ocean, em_ctrl, em_ctrl_land, em_ctrl_ocean, em_sens, em_sens_land, em_sens_ocean):
    for df_s, df_c, em_s, em_c, title, dom in [
            (dset_sens, dset_ctrl, em_sens, em_ctrl, 'a) GLOBAL', 'global'),
            (dset_sens_land, dset_ctrl_land, em_sens_land, em_ctrl_land, 'b) LAND ONLY', 'land'),
            (dset_sens_ocean, dset_ctrl_ocean, em_sens_ocean, em_ctrl_ocean, 'c) OCEAN ONLY', 'ocean')]:
        _plot_panel(df_s, df_c, em_s, em_c, title, f'{FIG_DIR}/{var}_bias_{dom}_{season}_1999.png')

    # Carica le immagini
    from matplotlib.gridspec import GridSpec
    import matplotlib.image as mpimg
    img1 = mpimg.imread(f'{FIG_DIR}/tas_bias_global_{season}_1999.png')
    img2 = mpimg.imread(f'{FIG_DIR}/tas_bias_land_{season}_1999.png')
    img3 = mpimg.imread(f'{FIG_DIR}/tas_bias_ocean_{season}_1999.png')
    
    fig = plt.figure(figsize=(10, 8), constrained_layout=False)
    
    # Definisci il layout: 1 righe, 3 colonne, con GridSpec
    gs = GridSpec(1, 3, wspace=0.0, hspace=0.0)
    
    # Aggiungi i sottoplot
    axes = [fig.add_subplot(gs[0, i]) for i in range(3)]
    images = [img1, img2, img3]
    
    # Mostra immagini senza assi
    for ax, img in zip(axes, images):
        ax.imshow(img)
        ax.axis('off')
    # Applica layout compatto
    plt.tight_layout(pad=0.0)
    # Sistema il layout
    #plt.tight_layout()
    plt.savefig(f'{FIG_DIR}/fig3_tas_global_bias_{season}_1999.png', dpi=600, bbox_inches = 'tight')
    plt.close('all')

In [ ]:
%%time
seasons = ['DJF','MAM','JJA','SON']
futures = []

# Submit all (season, lead) combinations in parallel
with concurrent.futures.ProcessPoolExecutor() as executor:
    futures = [executor.submit(main_script, season) for season in seasons]
    # progresso: stampa ogni task man mano che termina
    for _i, _f in enumerate(concurrent.futures.as_completed(futures), 1):
        _f.result()
        print(f"  completato {_i}/{len(futures)}", flush=True)

In [ ]:
# ============================================================================
# Bias GLOBALE per lead time + combinazioni di lead year, un boxplot per stagione
# (stile Fig3/Fig5: box affiancati SENS/CTRL, medie d'ensemble centrate e spezzate
# tra i gruppi). Sostituisce, per il bias globale, i 15 grafici box-per-anno di
# 05-Serie_temporale_anomalie con un'unica figura per stagione.
#
# Significativita' codificata nella trasparenza del colore (non con un
# asterisco): box pieno/opaco se SENS e CTRL differiscono significativamente
# a quella posizione, sbiadito altrimenti - stesso schema applicato alla
# versione annuale (03-Fig3_global_bias_tas.ipynb). significant_labels e
# _draw_significance_boxplot stanno nella cella prima di plot_figures,
# condivise coi pannelli per lead singolo.
# ============================================================================
import itertools
import numpy as np
import matplotlib.patches as mpatches
from scipy import stats

seasons = ['DJF', 'MAM', 'JJA', 'SON']  # ridefinita qui: la cella indipendente non richiede la vecchia cella 5

LABELS = ['0', '1', '2', '3', '4',
          '0-1', '1-2', '2-3', '3-4',
          '0-2', '1-3', '2-4',
          '0-3', '1-4',
          '0-4']

def _win(lab):
    if '-' in lab:
        a, b = lab.split('-')
        return int(b) - int(a) + 1
    return 1

def _bias_file(exp, label, season):
    tag = f"leadyears_{label}" if '-' in label else f"lead_{label}"
    return f"{SAVE_PATH}/{exp}_{var}_{tag}_1x1_global_average_all_members_{season}_1999.nc"

def load_global_bias(season):
    import glob
    rows, em_s, em_c, used = [], {}, {}, []
    for lab in LABELS:
        fs, fc = _bias_file(exp_sens, lab, season), _bias_file(exp_ctrl, lab, season)
        if not (glob.glob(fs) and glob.glob(fc)):
            print(f"[{season}] manca lead/combo {lab}, salto")
            continue
        used.append(lab)
        vs = xr.open_dataset(fs)[var].values.ravel()
        vc = xr.open_dataset(fc)[var].values.ravel()
        rows += [(lab, float(v), 'SENS') for v in vs]
        rows += [(lab, float(v), 'CTRL') for v in vc]
        em_s[lab] = float(np.nanmean(vs))
        em_c[lab] = float(np.nanmean(vc))
    return pd.DataFrame(rows, columns=['lead_label', var, 'exp']), used, em_s, em_c

def plot_bias_leadyears(season):
    df, used, em_s, em_c = load_global_bias(season)
    if df.empty:
        print(f"[{season}] nessun dato disponibile, salto il plot")
        return
    ymin, ymax = df[var].min(), df[var].max()
    margin = 0.10 * (ymax - ymin)
    ylim = (ymin - margin, ymax + margin)

    fig, ax = plt.subplots(figsize=(22, 6))
    positions = np.arange(len(used))
    sig = significant_labels(df, 'lead_label', used)
    _draw_significance_boxplot(ax, positions, used, df, 'lead_label', sig)

    off = 0.2
    for _w, _grp in itertools.groupby(range(len(used)), key=lambda i: _win(used[i])):
        idxs = np.array(list(_grp))
        ax.plot(idxs - off, [em_s[used[i]] for i in idxs], color='k')
        ax.plot(idxs + off, [em_c[used[i]] for i in idxs], color='k', linestyle='--')

    ax.set_ylim(*ylim)
    ax.set_ylabel(f'{var} bias (K)')
    ax.set_xlabel('lead years')
    ax.set_title(f'Bias globale - {season}')
    plt.setp(ax.get_xticklabels(), rotation=45)

    fig.savefig(f'{FIG_DIR}/{var}_bias_leadyears_{season}_1999.png', dpi=600, bbox_inches='tight')
    plt.close('all')


In [ ]:
%%time
# un boxplot per stagione (4 in tutto); plotting leggero, esecuzione seriale
for season in seasons:
    plot_bias_leadyears(season)
    print(f"  {season} completata", flush=True)
